# 💳 Credit Card Fraud Detection

## CodSoft Data Science Internship — Task 5

---

**Objective:** Build a machine learning model to identify fraudulent credit card transactions using classification algorithms on a highly imbalanced dataset.

**Dataset:** [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

**Models:** Logistic Regression, Decision Tree, Random Forest, Gradient Boosting

---

## 1. Setup & Imports

In [ ]:
# Standard Libraries
import sys
import warnings
from pathlib import Path

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, auc
)
import joblib

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 200)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 12
})

print('✅ All libraries imported successfully!')
print(f'📂 Project root: {project_root}')

In [ ]:
# Create directories
output_dir = project_root / 'outputs'
model_dir = project_root / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
print('✅ Directories created/verified')

## 2. Load Dataset

In [ ]:
# Load the credit card fraud dataset
dataset_path = project_root / 'dataset' / 'creditcard.csv'

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}\n"
        "Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud"
    )

df = pd.read_csv(dataset_path)
print(f'✅ Dataset loaded successfully!')
print(f'📊 Shape: {df.shape}')
print(f'💾 Memory: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB')

## 3. Dataset Overview

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Column names and data types
print('Columns:', list(df.columns))
print(f'\nTotal columns: {len(df.columns)}')
print(f'\nData types:\n{df.dtypes.value_counts()}')

## 4. Missing Value Analysis

In [ ]:
# Check for missing values
missing = df.isnull().sum()
total_missing = missing.sum()

print(f'Total missing values: {total_missing}')

if total_missing == 0:
    print('✅ No missing values found!')
else:
    print('⚠️ Missing values detected:')
    print(missing[missing > 0])

In [ ]:
# Missing values heatmap
fig, ax = plt.subplots(figsize=(14, 6))
missing_data = df.isnull().sum().values.reshape(1, -1)
sns.heatmap(missing_data, yticklabels=['Missing Count'], xticklabels=df.columns,
            cmap='YlOrRd', annot=True, fmt='d', ax=ax, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Missing Values Heatmap', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / 'missing_values_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: missing_values_heatmap.png')

## 5. Duplicate Detection

In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f'Duplicate rows: {duplicate_count}')

if duplicate_count > 0:
    print(f'Removing {duplicate_count} duplicates...')
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'New shape: {df.shape}')
else:
    print('✅ No duplicates found!')

## 6. Class Distribution Analysis

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
class_pcts = df['Class'].value_counts(normalize=True) * 100

print('--- Class Distribution ---')
print(f'Legitimate (0): {class_counts[0]:,} ({class_pcts[0]:.3f}%)')
print(f'Fraud (1):      {class_counts[1]:,} ({class_pcts[1]:.3f}%)')
print(f'Imbalance Ratio: {class_counts[0] / class_counts[1]:.1f}:1')

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2563EB', '#EF4444']
labels = ['Legitimate', 'Fraud']

axes[0].bar(labels, class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=11, fontweight='bold')

axes[1].pie(class_counts.values, labels=labels, colors=colors,
            autopct='%1.3f%%', startangle=90,
            textprops={'fontsize': 12}, explode=(0, 0.1))
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('Transaction Class Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: class_distribution.png')

## 7. Exploratory Data Analysis

In [ ]:
# Transaction Amount Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Class'] == 0]['Amount'], bins=80, color='#2563EB',
             alpha=0.7, label='Legitimate', edgecolor='white')
axes[0].hist(df[df['Class'] == 1]['Amount'], bins=80, color='#EF4444',
             alpha=0.7, label='Fraud', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Amount ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].set_xlim(0, 500)

axes[1].hist(np.log1p(df[df['Class'] == 0]['Amount']), bins=60, color='#2563EB',
             alpha=0.7, label='Legitimate', edgecolor='white')
axes[1].hist(np.log1p(df[df['Class'] == 1]['Amount']), bins=60, color='#EF4444',
             alpha=0.7, label='Fraud', edgecolor='white')
axes[1].set_title('Log Transaction Amount Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Amount + 1)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig(output_dir / 'transaction_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: transaction_amount_distribution.png')

In [ ]:
# Fraud vs Legitimate Transactions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

legitimate = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

axes[0, 0].boxplot([legitimate, fraud], labels=['Legitimate', 'Fraud'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#2563EB', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
axes[0, 0].set_title('Amount: Legitimate vs Fraud', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Amount ($)', fontsize=11)
axes[0, 0].set_ylim(0, 500)

axes[0, 1].hist(df[df['Class'] == 0]['Time'], bins=48, color='#2563EB',
                 alpha=0.6, label='Legitimate', density=True, edgecolor='white')
axes[0, 1].hist(df[df['Class'] == 1]['Time'], bins=48, color='#EF4444',
                 alpha=0.6, label='Fraud', density=True, edgecolor='white')
axes[0, 1].set_title('Time Distribution by Class', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Time (seconds)', fontsize=11)
axes[0, 1].set_ylabel('Density', fontsize=11)
axes[0, 1].legend(fontsize=10)

for idx, v_col in enumerate(['V14', 'V17']):
    ax = axes[1, idx]
    ax.hist(df[df['Class'] == 0][v_col], bins=60, color='#2563EB',
            alpha=0.6, label='Legitimate', density=True, edgecolor='white')
    ax.hist(df[df['Class'] == 1][v_col], bins=60, color='#EF4444',
            alpha=0.6, label='Fraud', density=True, edgecolor='white')
    ax.set_title(f'{v_col} Distribution by Class', fontsize=13, fontweight='bold')
    ax.set_xlabel(v_col, fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Fraud vs Legitimate Transactions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(output_dir / 'fraud_vs_legitimate.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fraud_vs_legitimate.png')

In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(16, 12))

selected_cols = ['V1', 'V2', 'V3', 'V4', 'V5', 'V10', 'V11', 'V12',
                 'V14', 'V16', 'V17', 'V18', 'Time', 'Amount', 'Class']
corr_matrix = df[selected_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap (Selected Features)', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: correlation_heatmap.png')

## 8. Data Preprocessing

In [ ]:
# Train/Test Split
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')
print(f'\nTrain fraud ratio: {y_train.mean():.4f}')
print(f'Test fraud ratio:  {y_test.mean():.4f}')

In [ ]:
# Feature Scaling
scaler = StandardScaler()
columns_to_scale = ['Time', 'Amount']

X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

# Save scaler
joblib.dump(scaler, model_dir / 'scaler.pkl')
print('✅ Features scaled and scaler saved')

## 9. Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features

X_train, X_test, new_features = engineer_features(X_train, X_test)

print(f'\nNew features added: {new_features}')
print(f'Total features: {X_train.shape[1]}')
print(f'\nTraining shape: {X_train.shape}')
print(f'Testing shape:  {X_test.shape}')

## 10. Model Training & Comparison

In [ ]:
import time

# Define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced', n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42, class_weight='balanced', max_depth=10
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1, max_depth=15
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1
    ),
}

# Train and evaluate all models
results = {}
comparison_data = []

for name, model in models.items():
    print(f'\n🔄 Training {name}...')
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1_Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else 0,
        'Training_Time_Sec': round(train_time, 2)
    }
    
    results[name] = {'model': model, 'predictions': y_pred, 'probabilities': y_prob, 'metrics': metrics}
    comparison_data.append(metrics)
    
    print(f'  ✅ Accuracy: {metrics["Accuracy"]:.4f} | Precision: {metrics["Precision"]:.4f} | '
          f'Recall: {metrics["Recall"]:.4f} | F1: {metrics["F1_Score"]:.4f} | '
          f'ROC-AUC: {metrics["ROC_AUC"]:.4f}')

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values(by='F1_Score', ascending=False).reset_index(drop=True)
comparison_df.to_csv(output_dir / 'model_comparison.csv', index=False)

print('\n--- Model Comparison ---')
comparison_df

## 11. Hyperparameter Tuning

In [ ]:
# Select best model by F1 score
best_model_name = comparison_df.iloc[0]['Model']
print(f'🏆 Best Model: {best_model_name}')
print(f'\nPerforming GridSearchCV...')

# Parameter grids
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs'],
    },
    'Decision Tree': {
        'max_depth': [5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
    },
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [10, 15, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
    },
}

base_model = models[best_model_name]
param_grid = param_grids[best_model_name]

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

print(f'\n✅ Best Parameters: {best_params}')
print(f'✅ Best CV F1 Score: {grid_search.best_score_:.4f}')

## 12. Model Evaluation

In [ ]:
# Evaluate tuned model
y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else None

print(f'=== Classification Report — Tuned {best_model_name} ===')
print(classification_report(y_test, y_pred_tuned, target_names=['Legitimate', 'Fraud']))

print(f'Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_tuned):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_tuned):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_tuned):.4f}')
if y_prob_tuned is not None:
    print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_tuned):.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_tuned)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'],
            ax=ax, annot_kws={'size': 14})
ax.set_title(f'Confusion Matrix — Tuned {best_model_name}', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
plt.tight_layout()
plt.savefig(output_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrix.png')

In [ ]:
# ROC Curve
if y_prob_tuned is not None:
    fpr, tpr, _ = roc_curve(y_test, y_prob_tuned)
    roc_auc_val = auc(fpr, tpr)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='#2563EB', lw=2.5,
            label=f'Tuned {best_model_name} (AUC = {roc_auc_val:.4f})')
    ax.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--',
            label='Random Classifier')
    ax.fill_between(fpr, tpr, alpha=0.1, color='#2563EB')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate', fontsize=13)
    ax.set_title('ROC Curve', fontsize=16, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: roc_curve.png')

In [ ]:
# Precision-Recall Curve
if y_prob_tuned is not None:
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob_tuned)
    pr_auc_val = auc(recall_vals, precision_vals)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall_vals, precision_vals, color='#7C3AED', lw=2.5,
            label=f'Tuned {best_model_name} (AUC = {pr_auc_val:.4f})')
    ax.fill_between(recall_vals, precision_vals, alpha=0.1, color='#7C3AED')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall', fontsize=13)
    ax.set_ylabel('Precision', fontsize=13)
    ax.set_title('Precision-Recall Curve', fontsize=16, fontweight='bold')
    ax.legend(loc='lower left', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'precision_recall_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: precision_recall_curve.png')

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    feature_names = list(X_test.columns)
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:20]
    
    top_features = [feature_names[i] for i in indices]
    top_importances = importances[indices]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
    ax.barh(range(len(top_features)), top_importances[::-1],
            color=colors[::-1], edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features[::-1], fontsize=10)
    ax.set_xlabel('Feature Importance', fontsize=13)
    ax.set_title(f'Top 20 Feature Importances — Tuned {best_model_name}',
                 fontsize=16, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: feature_importance.png')

## 13. Generate Predictions

In [ ]:
# Generate predictions CSV
predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred_tuned,
})

if y_prob_tuned is not None:
    predictions_df['Probability'] = np.round(y_prob_tuned, 6)

predictions_df['Correct'] = (predictions_df['Actual'] == predictions_df['Predicted']).astype(int)

predictions_df.to_csv(output_dir / 'predictions.csv', index=False)

total = len(predictions_df)
correct = predictions_df['Correct'].sum()

print(f'Total Predictions: {total}')
print(f'Correct: {correct}')
print(f'Incorrect: {total - correct}')
print(f'Accuracy: {correct/total:.4f}')
print(f'\n✅ Saved: predictions.csv')

predictions_df.head(10)

## 14. Save Model

In [ ]:
# Save the tuned model
model_path = model_dir / 'fraud_detection_model.pkl'
joblib.dump(best_model, model_path)

print(f'✅ Model saved to: {model_path}')
print(f'✅ Scaler saved to: {model_dir / "scaler.pkl"}')

# Verify
loaded_model = joblib.load(model_path)
verify_pred = loaded_model.predict(X_test[:5])
print(f'\n🔍 Verification predictions: {verify_pred}')
print('✅ Model loads and predicts correctly!')

## 15. Summary

In [ ]:
print('=' * 60)
print('  CREDIT CARD FRAUD DETECTION — COMPLETE')
print('  CodSoft Data Science Internship — Task 5')
print('=' * 60)

print(f'\n  Best Model: {best_model_name}')
print(f'  Best Parameters: {best_params}')

print(f'\n  Final Results:')
print(f'    Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'    Precision: {precision_score(y_test, y_pred_tuned):.4f}')
print(f'    Recall:    {recall_score(y_test, y_pred_tuned):.4f}')
print(f'    F1 Score:  {f1_score(y_test, y_pred_tuned):.4f}')
if y_prob_tuned is not None:
    print(f'    ROC-AUC:   {roc_auc_score(y_test, y_prob_tuned):.4f}')

print(f'\n  Generated Outputs:')
for f in sorted(output_dir.iterdir()):
    if f.name != '.gitkeep':
        print(f'    ✓ {f.name}')

print(f'\n  Saved Models:')
for f in sorted(model_dir.iterdir()):
    if f.name != '.gitkeep':
        print(f'    ✓ {f.name}')

print('\n' + '=' * 60)
print('  Pipeline Complete! 🎉')
print('=' * 60)